# Conditional Autoencoder for ETF Rotation

The CAE notebook runs the structural extractor and then maps checkpointed
factor states through the shared factor-premium baseline.

In [1]:
"""ETF CAE case-study run via the shared latent-factor library path."""

import warnings

from case_studies.utils.latent_factors.case_study import (
    configured_models,
    load_case_study_context,
    run_case_study_model,
)

warnings.filterwarnings("ignore")

In [2]:
CASE_STUDY_ID = "etfs"
PRIMARY_LABEL = ""
MAX_SYMBOLS = 0
N_FACTORS = 5
N_EPOCHS = 50
USE_CACHE = True
FORCE_RETRAIN = False
MAX_FOLDS = 0
MAX_VARIANT_LABELS = -1
USE_MACRO = False
MODEL_NAME = "cae"
REPORTING_CHECKPOINT = N_EPOCHS

In [4]:
context = load_case_study_context(
    CASE_STUDY_ID,
    primary_label=PRIMARY_LABEL,
    max_symbols=MAX_SYMBOLS,
    max_folds=MAX_FOLDS,
    max_variant_labels=MAX_VARIANT_LABELS,
    use_macro=USE_MACRO,
)
available_models = configured_models(context)
if MODEL_NAME not in available_models:
    raise ValueError(f"{MODEL_NAME!r} is not configured for {CASE_STUDY_ID}")

print(f"Case study: {CASE_STUDY_ID}")
print(f"Model: {MODEL_NAME}")
print(f"Primary label: {context.primary_label}")
print(f"Dataset rows: {len(context.dataset):,}")
print(f"Features: {len(context.feature_names)}")
print(f"Splits: {len(context.splits)}")
print(f"Model kwargs: {context.model_kwargs.get(MODEL_NAME, {})}")

Case study: etfs
Model: cae
Primary label: fwd_ret_21d
Dataset rows: 394,233
Features: 71
Splits: 8
Model kwargs: {'n_factors': 5, 'n_epochs': 50, 'checkpoint_interval': 5}


## Run Walk-Forward CV

In [5]:
result = run_case_study_model(
    context,
    model_name=MODEL_NAME,
    notebook="11c_conditional_autoencoder",
    n_factors=N_FACTORS,
    n_epochs=N_EPOCHS,
    use_cache=USE_CACHE,
    force_retrain=FORCE_RETRAIN,
    reporting_epoch=REPORTING_CHECKPOINT,
)

if result["model_results"][0]["best_epoch"] != REPORTING_CHECKPOINT:
    raise AssertionError("CAE result did not use the fixed reporting checkpoint")
print(result["model_results"])
print(result["fold_metrics"][MODEL_NAME])

Latent factor CV: 1 models × 8 folds
Log file: case_studies/etfs/run_log/latent_factors.log
Scoring: dates=rebalance cadence=monthly_month_end step=1 checkpoint_selection=fixed reporting_epoch=last
  cae (K=5):


    Fold 0: ragged train=2497, val=252, max_N=96


      fold 0: reported_epoch=0, IC=-0.0340, 11.5s


    Fold 1: ragged train=2497, val=252, max_N=96


      fold 1: reported_epoch=0, IC=-0.0736, 10.3s


...

    Fold 6: ragged train=2493, val=252, max_N=94


      fold 6: reported_epoch=0, IC=+0.1730, 10.3s


    Fold 7: ragged train=2241, val=252, max_N=89


      fold 7: reported_epoch=0, IC=-0.1021, 9.4s


    -> best epoch=0, IC=+0.0315 (151.3s)
  Best: cae (IC=+0.0315)
[{'model_name': 'cae', 'mean_ic': 0.0315, 'best_epoch': 0, 'n_folds': 8, 'elapsed_s': 151.3, 'started_at': '2026-05-23T11:46:32.728863+00:00'}]
shape: (88, 6)
┌─────────┬───────┬─────────┬─────────┬────────┬────────────────┐
│ fold_id ┆ epoch ┆ ic_mean ┆ n_train ┆ n_test ┆ n_scored_dates │
│ ---     ┆ ---   ┆ ---     ┆ ---     ┆ ---    ┆ ---            │
│ i64     ┆ i64   ┆ f64     ┆ i64     ┆ i64    ┆ i64            │
╞═════════╪═══════╪═════════╪═════════╪════════╪════════════════╡
│ 0       ┆ 0     ┆ -0.034  ┆ 2497    ┆ 252    ┆ 13             │
│ 0       ┆ 5     ┆ -0.0592 ┆ 2497    ┆ 252    ┆ 13             │
│ 0       ┆ 10    ┆ -0.0634 ┆ 2497    ┆ 252    ┆ 13             │
│ 0       ┆ 15    ┆ -0.0432 ┆ 2497    ┆ 252    ┆ 13             │
│ 0       ┆ 20    ┆ -0.0364 ┆ 2497    ┆ 252    ┆ 13             │
│ …       ┆ …     ┆ …       ┆ …       ┆ …      ┆ …              │
│ 7       ┆ 30    ┆ -0.1067 ┆ 2241    ┆ 252    ┆ 